# Notebook 3 — Feature Extraction

Extract structural features from the MD trajectory.  
The canonical feature set for committor regression uses:
1. **Pairwise O/N distances** — distances between all heavy O and N atoms  
   (hydrogen-bond relevant, identified by `ONH.py`).
2. **Backbone torsions** — φ/ψ angles in cos/sin form.

Both are extracted at the same stride so they align frame-by-frame  
with the cluster assignment CSV.

**Output:** DataFrames used directly in Notebook 4 (regression).

In [ ]:
import yaml
import sys
sys.path.insert(0, "..")

from src.ONH import ONH
from src.generate_pair import pair
from src.extract_features import features

In [ ]:
# --- Select system ---
with open("../config/chignolin.yaml") as f:   # change to ww_domain.yaml for WW domain
    cfg = yaml.safe_load(f)

PDB    = cfg["topology_pdb"]
FILES  = cfg["trajectory_xtc"]
STRIDE = cfg["stride"]

print(f"PDB:    {PDB}")
print(f"Stride: {STRIDE}")

## Step 1 — Identify O/N atoms and generate distance pairs

In [ ]:
# Identify all O/N heavy atoms in the PDB
A, B = ONH(PDB)
print(f"O/N atoms:      {len(A)}")
print(f"Polar-H bonded: {len(B)}")

# Generate all unique pairs (0-based indices for PyEMMA)
atom_pairs = pair(A, B)
print(f"Atom pairs:     {len(atom_pairs)}")

## Step 2 — Extract features

In [ ]:
# Pairwise O/N distances
df_dist = features(PDB, FILES, pairs=atom_pairs, al="custom_distances", stride=STRIDE)
print(f"Distance features shape: {df_dist.shape}")
df_dist.head(3)

In [ ]:
# Backbone torsions (φ/ψ in cos/sin form)
df_tors = features(PDB, FILES, al="backbone_torsions", stride=STRIDE)
print(f"Torsion features shape: {df_tors.shape}")
df_tors.head(3)

## Optional: Save feature matrices to disk

If working interactively, re-extracting features takes several minutes.  
You can cache them here and load in Notebook 4.

In [ ]:
import os
SYSTEM = cfg["system"]
out_dir = f"../data/{SYSTEM}"
os.makedirs(out_dir, exist_ok=True)

df_dist.to_csv(f"{out_dir}/distances_ON.csv", index=False)
df_tors.to_csv(f"{out_dir}/backbone_torsions.csv", index=False)
print("Feature CSVs saved.")